# BirdCLEF+ 2026 — Submission (CPU, PyTorch)

Kaggle CPU notebook. **No internet, <= 90 min, CPU only.**

Dependencies (pre-installed on every Kaggle image): `numpy`, `pandas`, `soundfile`, `librosa`, `torch`, `timm`.

Model weights are a PyTorch checkpoint (`best.pt`) shipped via your Kaggle Model / Dataset.

Pipeline:
1. Load each soundscape with `soundfile` in a thread pool.
2. Split into 5-sec segments.
3. Batched log-mel (NumPy).
4. PyTorch CPU inference (EfficientNet-B0).
5. Temporal smoothing + file-level prior.
6. Write `submission.csv`.

In [ ]:
import os, time, glob
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import timm

torch.set_num_threads(os.cpu_count() or 4)
torch.set_grad_enabled(False)


def _find_one(patterns, must_contain):
    for pat in patterns:
        for hit in glob.glob(pat):
            p = Path(hit)
            if must_contain == '.' or (p / must_contain).exists():
                return p
    return None


# ---- constants (MUST match training) --------------------------------------
SR = 32_000
CLIP_SAMPLES = 5 * SR
N_FFT, HOP, WIN = 1024, 320, 1024
N_MELS, FMIN, FMAX = 128, 20, 16_000
TOP_DB = 80.0
BACKBONE = 'tf_efficientnet_b0.ns_jft_in1k'

COMP_DIR = _find_one(
    [
        '/kaggle/input/birdclef-2026',
        '/kaggle/input/competitions/birdclef-2026',
        '/kaggle/input/*birdclef*2026*',
    ],
    must_contain='taxonomy.csv',
)
assert COMP_DIR is not None, 'birdclef-2026 competition dataset not attached.'
TEST_DIR = COMP_DIR / 'test_soundscapes'

# Find best.pt anywhere under /kaggle/input (Kaggle Model or Dataset).
CKPT_CANDIDATES = sorted(glob.glob('/kaggle/input/**/best.pt', recursive=True))
assert CKPT_CANDIDATES, 'no best.pt found under /kaggle/input — attach the model.'
CKPT_PATH = Path(CKPT_CANDIDATES[0])
OUT_PATH = Path('submission.csv')

tax = pd.read_csv(COMP_DIR / 'taxonomy.csv')
CLASSES = tax['primary_label'].tolist()
NC = len(CLASSES)
print('data root:', COMP_DIR)
print('checkpoint:', CKPT_PATH)
print('classes:', NC)

MEL_FB = librosa.filters.mel(sr=SR, n_fft=N_FFT, n_mels=N_MELS, fmin=FMIN, fmax=FMAX).astype(np.float32)


In [ ]:
def load_audio(path):
    y, sr = sf.read(str(path), dtype='float32', always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr != SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR)
    return y.astype(np.float32, copy=False)

def segment_file(y, n=CLIP_SAMPLES):
    total = int(np.ceil(len(y) / n)) * n
    if len(y) < total:
        y = np.pad(y, (0, total - len(y)))
    return y.reshape(-1, n)

def logmel_batch(segs):
    # segs: (B, CLIP_SAMPLES) -> (B, 1, N_MELS, T)
    out = np.empty((segs.shape[0], N_MELS, CLIP_SAMPLES // HOP + 1), dtype=np.float32)
    for i, s in enumerate(segs):
        S = librosa.stft(s, n_fft=N_FFT, hop_length=HOP, win_length=WIN, center=True)
        P = (S.real ** 2 + S.imag ** 2).astype(np.float32)
        M = MEL_FB @ P
        out[i] = librosa.power_to_db(M, top_db=TOP_DB)
    m = out.mean(axis=(1, 2), keepdims=True)
    sd = out.std(axis=(1, 2), keepdims=True) + 1e-6
    out = (out - m) / sd
    return out[:, None, :, :]

In [ ]:
# ---- Build model and load best.pt ----------------------------------------
model = timm.create_model(
    BACKBONE,
    pretrained=False,
    in_chans=1,
    num_classes=0,
    global_pool='avg',
    drop_rate=0.0,
)
feat_dim = model.num_features
head = torch.nn.Sequential(
    torch.nn.Dropout(0.0),
    torch.nn.Linear(feat_dim, NC),
)

ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
state = ckpt['model'] if isinstance(ckpt, dict) and 'model' in ckpt else ckpt

# state_dict was saved with prefixes 'backbone.' and 'head.' (TimmMelClassifier).
backbone_sd = {k.removeprefix('backbone.'): v for k, v in state.items() if k.startswith('backbone.')}
head_sd     = {k.removeprefix('head.'):     v for k, v in state.items() if k.startswith('head.')}
missing_b, unexpected_b = model.load_state_dict(backbone_sd, strict=False)
missing_h, unexpected_h = head.load_state_dict(head_sd, strict=False)
print('backbone missing:', len(missing_b), 'unexpected:', len(unexpected_b))
print('head missing:', len(missing_h), 'unexpected:', len(unexpected_h))

model.eval(); head.eval()
if isinstance(ckpt, dict) and 'auc' in ckpt:
    print('val AUC at checkpoint:', ckpt['auc'])


In [ ]:
# ---- Optional Model B disabled in this PyTorch-only build. ----------------
USE_MODEL_B = False
EMB_WEIGHT = 0.0


In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

@torch.inference_mode()
def predict_mels(mels: np.ndarray) -> np.ndarray:
    """mels: (B, 1, N_MELS, T) float32 -> probs (B, NC) float32."""
    x = torch.from_numpy(mels)
    f = model(x)
    logits = head(f)
    return torch.sigmoid(logits).cpu().numpy().astype(np.float32)


In [ ]:
files = sorted(TEST_DIR.glob('*.ogg'))
print('test files:', len(files))
if not files:
    print('NOTE: test_soundscapes is empty in interactive runs — real audio is only')
    print('      mounted during Kaggle scoring. The next cell will write a header-only')
    print('      submission.csv; submit the version anyway and Kaggle will rerun it on')
    print('      the hidden test set.')

row_ids = []
all_probs = []

t0 = time.time()
with ThreadPoolExecutor(max_workers=min(4, os.cpu_count() or 2)) as pool:
    preloaded = {p: pool.submit(load_audio, p) for p in files[: min(4, len(files))]}
    for idx, p in enumerate(files):
        look_ahead = 4
        for q in files[idx + 1 : idx + 1 + look_ahead]:
            if q not in preloaded:
                preloaded[q] = pool.submit(load_audio, q)
        y = preloaded.pop(p).result()

        segs = segment_file(y)
        mels = logmel_batch(segs).astype(np.float32)

        probs = predict_mels(mels)

        # Temporal smoothing (3-tap 0.2/0.6/0.2).
        if probs.shape[0] >= 3:
            pad = np.pad(probs, ((1, 1), (0, 0)), mode='edge')
            probs = 0.2 * pad[:-2] + 0.6 * pad[1:-1] + 0.2 * pad[2:]
        # File-level prior.
        probs = probs * (0.8 + 0.2 * probs.max(axis=0, keepdims=True))

        stem = p.stem
        for i in range(probs.shape[0]):
            end_sec = (i + 1) * 5
            row_ids.append(f'{stem}_{end_sec}')
        all_probs.append(probs)

        if (idx + 1) % 50 == 0:
            print(f'{idx+1}/{len(files)}   elapsed {time.time()-t0:.1f}s')

print('done. files:', len(files), 'rows:', len(row_ids), 'elapsed:', f'{time.time()-t0:.1f}s')


In [ ]:
if all_probs:
    arr = np.concatenate(all_probs, axis=0)
    sub = pd.DataFrame(arr, columns=CLASSES)
    sub.insert(0, 'row_id', row_ids)
else:
    # Interactive run: test_soundscapes is empty (real files appear only during
    # Kaggle's scoring rerun). Write a header-only csv so the submission is valid.
    print('no test files visible — writing header-only submission.csv (this is normal in interactive runs).')
    sub = pd.DataFrame(columns=['row_id'] + CLASSES)

sub.to_csv(OUT_PATH, index=False, float_format='%.5f')
print('wrote', OUT_PATH, sub.shape)
sub.head()
